# 00 - Test connexion S3 ↔ Spark


In [ ]:
import os
from pyspark.sql import SparkSession

# == CREDENTIALS S3 - NE PAS PUSHER SUR GIT ==
# Demander les valeurs à Manar (Architecte Data)
os.environ['AWS_ACCESS_KEY_ID']     = 'METTRE_ICI'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'METTRE_ICI'
os.environ['AWS_SESSION_TOKEN']     = 'METTRE_ICI'
# =============================================

spark = SparkSession.builder \
    .appName('test-connexion-s3') \
    .config('spark.hadoop.fs.s3a.endpoint', 'https://minio.lab.sspcloud.fr') \
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID']) \
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY']) \
    .config('spark.hadoop.fs.s3a.session.token', os.environ['AWS_SESSION_TOKEN']) \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider') \
    .config('spark.hadoop.fs.s3a.path.style.access', 'true') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .getOrCreate()

print(f'✓ Spark {spark.version} démarré')

In [ ]:
# Lecture Bronze
df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv('s3a://manar1305/data-factory-bronze/accidents/US_Accidents_March23.csv')

print(f'✓ Lignes   : {df.count()}')        # attendu : 7 728 394
print(f'✓ Colonnes : {len(df.columns)}')   # attendu : 46
df.printSchema()
df.show(5)

In [ ]:
# Vérification Silver (données nettoyées par les Data Engineers)
df_silver = spark.read.parquet('s3a://manar1305/data-factory-silver/accidents/')
print(f' Silver — Lignes   : {df_silver.count()}')
print(f' Silver — Colonnes : {len(df_silver.columns)}')
df_silver.show(5)